# Baseline Model Comparison

## Purpose
Compare performance of multiple baseline models on the insurance claims dataset:
- **Linear Models**: LogisticRegression, Ridge
- **Tree Models**: RandomForest, XGBoost, ExtraTrees
- **Naive Models**: BernoulliNB

## Input Data
`insurance_claims_preprocessed_no_hobbies.csv` - Preprocessed dataset without feature engineering
`preprocesed_for_trees.csv` - Data optimized for tree models

## Output
- Baseline performance metrics for all models
- Model comparison (ROC-AUC, PR-AUC)
- Feature importance analysis

## Next Steps
- Feature engineering for linear models -> `03_feature_engineering_linear.ipynb`
- Hyperparameter tuning -> `04_logreg_tuning.ipynb`, `05_tree_model_tuning.ipynb`


In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Local Utils
from utils.data_loader import load_insurance_data
from utils.preprocessing import get_preprocessor
from utils.evaluation import evaluate_model


### Load Datasets
We load two versions of the dataset to see if tree-specific preprocessing (bin removal) helps.
- **Original**: Standard preprocessing (RobustScaler, OHE)
- **Trees**: No binning, kept continuous values


In [ ]:
datasets = {
    'Original': load_insurance_data('preprocessed', verbose=True),
    'Trees': load_insurance_data('trees', verbose=True)
}

# Containers for results
all_results = []
all_best_models = {}


### Define Baseline Models
We test a diverse set of algorithms using `RandomizedSearchCV` to find a decent baseline configuration for each.


In [ ]:
# 1. Linear & Naive Models
models_linear = [
    ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42, solver='saga'), {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__penalty': ['l1', 'l2']
    }),
    ('RidgeClassifier', RidgeClassifier(random_state=42), {
        'clf__alpha': [0.1, 1.0, 10.0]
    }),
    ('BernoulliNB', BernoulliNB(), {
        'clf__alpha': [0.1, 0.5, 1.0],
        'clf__binarize': [0.0, 0.5]
    })
]

# 2. Tree Models (RF, XGB, ET)
models_trees = [
    ('RandomForest', RandomForestClassifier(random_state=42), {
        'clf__n_estimators': [50, 100],
        'clf__max_depth': [None, 10, 20],
        'clf__min_samples_split': [2, 5]
    }),
    ('ExtraTrees', ExtraTreesClassifier(random_state=42), {
        'clf__n_estimators': [50, 100],
        'clf__max_depth': [None, 10, 20]
    }),
    ('XGBoost', XGBClassifier(random_state=42, eval_metric='logloss'), {
        'clf__n_estimators': [50, 100],
        'clf__learning_rate': [0.01, 0.1],
        'clf__max_depth': [3, 6]
    })
]

all_models = models_linear + models_trees



### Quick Check: LazyPredict
Before running our deep custom loop, let's just run a quick `LazyClassifier` scan to see which families of models look promising on the 'Trees' dataset.
This is a 'sanity check' to ensure we haven't missed a simple model that performs oddly well.


In [ ]:
# Optional: LazyPredict scan
try:
    import tqdm.notebook
    import tqdm
    # Hack: Force tqdm to use standard progress bar to avoid AttributeError: 'tqdm_notebook' object has no attribute 'disp'
    tqdm.notebook.tqdm = tqdm.tqdm
except ImportError:
    pass

try:
    from lazypredict.Supervised import LazyClassifier
    
    print("Running LazyPredict on 'Trees' dataset...")
    # We use the 'Trees' dataset (no OHE, minimal processing) for the check
    # Note: datasets dictionary must be loaded from previous cells
    if 'Trees' in datasets:
        d_lazy = datasets['Trees']
        X_lz = d_lazy.drop('target', axis=1)
        y_lz = d_lazy['target']
        
        X_train_lz, X_test_lz, y_train_lz, y_test_lz = train_test_split(X_lz, y_lz, test_size=0.2, stratify=y_lz, random_state=42)
        
        clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
        models_summary, predictions = clf.fit(X_train_lz, X_test_lz, y_train_lz, y_test_lz)
        
        print(models_summary)
    else:
        print("Trees dataset not found in 'datasets' dictionary.")
except ImportError as e:
    print(f"LazyPredict ImportError: {e}")
    print("Ensure all dependencies (lazypredict, xgboost, lightgbm) are installed.")
except Exception as e:
    print(f"LazyPredict Error: {e}")

In [ ]:
# Visualization of Top 5 Models from LazyPredict
if 'models_summary' in locals():
    plt.figure(figsize=(10, 6))
    # LazyPredict returns a DataFrame with models as index. Reset index to use 'Model' column for plotting
    # We focus on the top 5 models identifying by LazyPredict default sorting (usually F1/Accuracy/ROC combined or just F1)
    top_models = models_summary.head(5).reset_index()

    # Plot F1 Score and ROC AUC for comparison
    # Melding for easy seaborn plotting if we want multiple bars, but simple barplot is clearer for top 5
    sns.barplot(x='F1 Score', y='Model', data=top_models, palette='viridis')
    plt.title('Top 5 Models from LazyPredict (F1 Score)')
    plt.xlabel('F1 Score')
    plt.xlim(0, 1)  # Scores are usually 0-1
    plt.tight_layout()
    plt.show()
    
    display(top_models[['Model', 'F1 Score', 'ROC AUC', 'Accuracy']])
else:
    print("models_summary not found. Please run the LazyPredict cell above.")

### Training Loop
Train every model on both datasets.
- Use **StratifiedKFold** for robustness.
- Optimize for **Average Precision (PR-AUC)** since classes are imbalanced.


In [ ]:
for dname, df in datasets.items():
    print(f"\n{'='*40}\nProcessing Dataset: {dname}\n{'='*40}")
    
    X = df.drop('target', axis=1)
    y = df['target']
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # Preprocessor (RobustScaler for linear, but trees handles raw better - we use same prep for consistency here)
    # Note: For 'Trees' dataset, we might technically skip scaling, but RobustScaler is safe for trees too.
    preprocessor = get_preprocessor(X_train, strategy='robust')
    
    for name, model, grid in all_models:
        print(f"Training {name}...")
        
        # Pipeline
        pipe = Pipeline([('prep', preprocessor), ('clf', model)])
        
        # Search
        search = RandomizedSearchCV(
            pipe, grid, n_iter=5, scoring='average_precision',
            cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
            n_jobs=-1, random_state=42, verbose=0
        )
        
        try:
            search.fit(X_train, y_train)
            print(f"  -> Best CV Score: {search.best_score_:.4f}")
            
            best_model = search.best_estimator_
            
            # Evaluate on Test
            metrics = evaluate_model(best_model, X_test, y_test, model_name=name)
            metrics['dataset'] = dname
            metrics['model'] = name
            
            all_results.append(metrics)
            
            if dname not in all_best_models: all_best_models[dname] = {}
            all_best_models[dname][name] = best_model
            
        except Exception as e:
            print(f"  -> Failed: {e}")


### Final Comparison
Aggregate all results into a sorted table to find the winner.


In [ ]:
results_df = pd.DataFrame(all_results)

# Clean up columns for display
cols = ['dataset', 'model', 'pr_auc', 'roc_auc', 'f1', 'precision', 'recall']
display_df = results_df[cols].sort_values(['dataset', 'pr_auc'], ascending=[True, False])

print("=== Baseline Comparison Results ===")
print(display_df.to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x='model', y='pr_auc', hue='dataset', palette='viridis')
plt.title("Model Performance (PR-AUC) by Dataset")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
